# Institution-held-out validation (reviewer response)

**Purpose.** Leave-campus-out transportability check using `GroupKFold` grouped by `schoolnum`.

**Design (locked)**
- Models: CatBoost, XGBoost
- Feature sets: E1_full (25 features), E6_no_institution (drop `inst_*`)
- Split: `GroupKFold(n_splits=5)`, groups = campus ID (`schoolnum`)
- Preprocessing: median impute + StandardScaler fit on **train schools only** each fold
- Primary student-level results are **kept**; this notebook adds a sensitivity table

Outputs are written to `outputs/strengthened/`.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Match published student-level training budget (N_TREES=60 when XAI_FAST=1)
os.environ.setdefault("XAI_FAST", "1")

SCRIPT_DIR = Path.cwd()
if not (SCRIPT_DIR / "pipeline_common.py").exists():
    SCRIPT_DIR = Path(r"c:/Users/MossabBatal/Desktop/psychology_mas/xai_rag_depression")
    os.chdir(SCRIPT_DIR)

if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

from pipeline_common import (
    ALL_FEATURES,
    EXPERIMENT_FEATURE_SETS,
    OUTPUT_DIR,
    SEED,
    TARGET,
    build_ml_models,
    read_hms_csv,
    set_seeds,
)

set_seeds(SEED)
OUT = OUTPUT_DIR / "strengthened"
OUT.mkdir(parents=True, exist_ok=True)
DATA_CSV = SCRIPT_DIR.parent / "HMS_2024-2025_PUBLIC_instchars.csv"
N_SPLITS = 5
MODELS = ["CatBoost", "XGBoost"]
FEATURE_EXPERIMENTS = ["E1_full", "E6_no_institution"]
print("cwd:", Path.cwd())
print("data:", DATA_CSV.exists(), DATA_CSV)
print("out:", OUT)


## 1. Load analytic cohort + `schoolnum`

Same filters as `load_hms_dataframe`: Finished=1, non-missing outcome, ≥60% feature completeness.
`schoolnum` is retained only as the **grouping key** (never as a model feature).


In [ ]:
def load_cohort_with_schoolnum(features: list[str]) -> pd.DataFrame:
    raw = read_hms_csv(DATA_CSV)
    need = features + [TARGET, "nrweight", "Finished", "schoolnum"]
    missing = [c for c in need if c not in raw.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")
    df = raw[need].copy()
    for col in need:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df[df["Finished"] == 1].copy()
    df = df.dropna(subset=[TARGET, "schoolnum"])
    df[TARGET] = df[TARGET].astype(int)
    df["schoolnum"] = df["schoolnum"].astype(int)
    min_features = int(0.6 * len(features))
    df = df[df[features].notna().sum(axis=1) >= min_features].copy()
    df["nrweight"] = df["nrweight"].fillna(1.0).astype(float)
    return df.reset_index(drop=True)


df_full = load_cohort_with_schoolnum(ALL_FEATURES)
n_schools = int(df_full["schoolnum"].nunique())
print(f"N students={len(df_full):,} | N schools={n_schools} | prevalence={df_full[TARGET].mean():.3f}")
print(df_full.groupby("schoolnum").size().describe().round(1))


## 2. Institution-held-out GroupKFold

For each fold: fit imputer/scaler/model on train campuses → score entirely held-out campuses.


In [ ]:
def metrics_dict(y_true, y_prob, threshold: float = 0.5) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "brier": float(brier_score_loss(y_true, y_prob)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
    }


def run_groupkfold(df: pd.DataFrame, features: list[str], model_name: str, n_splits: int = N_SPLITS) -> list[dict]:
    X = df[features].values
    y = df[TARGET].values.astype(int)
    w = df["nrweight"].values.astype(float)
    groups = df["schoolnum"].values
    gkf = GroupKFold(n_splits=n_splits)
    rows = []
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=groups), start=1):
        pre = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
        X_tr = pre.fit_transform(X[tr]).astype(np.float32)
        X_te = pre.transform(X[te]).astype(np.float32)
        y_tr, y_te = y[tr], y[te]
        w_tr = w[tr]
        scale_pos = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
        clf = build_ml_models(scale_pos, seed=SEED)[model_name]
        clf.fit(X_tr, y_tr, sample_weight=w_tr)
        proba = clf.predict_proba(X_te)[:, 1]
        m = metrics_dict(y_te, proba)
        train_schools = set(groups[tr].tolist())
        test_schools = set(groups[te].tolist())
        assert train_schools.isdisjoint(test_schools), "Campus leakage across folds"
        rows.append({
            "fold": fold,
            "n_train": int(len(tr)),
            "n_test": int(len(te)),
            "n_train_schools": len(train_schools),
            "n_test_schools": len(test_schools),
            "test_prevalence": float(y_te.mean()),
            **m,
        })
        print(f"  fold {fold}: AUC={m['auc']:.4f} | test schools={len(test_schools)} | n_test={len(te):,}")
    return rows


per_fold_rows = []
for exp_name in FEATURE_EXPERIMENTS:
    feats = EXPERIMENT_FEATURE_SETS[exp_name]["features"]
    desc = EXPERIMENT_FEATURE_SETS[exp_name]["description"]
    print(f"\n=== {exp_name}: {desc} ({len(feats)} features) ===")
    for model_name in MODELS:
        print(f"-- {model_name}")
        fold_rows = run_groupkfold(df_full, feats, model_name)
        for r in fold_rows:
            per_fold_rows.append({
                "experiment": exp_name,
                "experiment_description": desc,
                "n_features": len(feats),
                "model": model_name,
                **r,
            })

per_fold = pd.DataFrame(per_fold_rows)
per_fold


## 3. Summary (mean ± SD across folds)

In [ ]:
metric_cols = ["auc", "pr_auc", "brier", "recall"]
summary = (
    per_fold.groupby(["experiment", "experiment_description", "n_features", "model"], as_index=False)
    .agg(
        n_folds=("fold", "count"),
        **{f"{c}_mean": (c, "mean") for c in metric_cols},
        **{f"{c}_std": (c, "std") for c in metric_cols},
    )
)
summary = summary.round(6)
summary


## 4. Comparison vs student-level holdout (primary results kept)

Student-level numbers come from the existing `outputs/model_performance.csv` (do **not** replace that table).


In [ ]:
student = pd.read_csv(OUTPUT_DIR / "model_performance.csv")
student = student.set_index("model")

cmp_rows = []
for _, row in summary.iterrows():
    model = row["model"]
    if model not in student.index:
        continue
    s_auc = float(student.loc[model, "auc"])
    s_pr = float(student.loc[model, "pr_auc"])
    s_brier = float(student.loc[model, "brier"])
    cmp_rows.append({
        "experiment": row["experiment"],
        "model": model,
        "student_level_auc": s_auc,
        "institution_heldout_auc_mean": row["auc_mean"],
        "institution_heldout_auc_std": row["auc_std"],
        "delta_auc_student_minus_heldout": s_auc - row["auc_mean"],
        "student_level_pr_auc": s_pr,
        "institution_heldout_pr_auc_mean": row["pr_auc_mean"],
        "institution_heldout_pr_auc_std": row["pr_auc_std"],
        "student_level_brier": s_brier,
        "institution_heldout_brier_mean": row["brier_mean"],
        "institution_heldout_brier_std": row["brier_std"],
    })

comparison = pd.DataFrame(cmp_rows).round(6)
display_cols = [
    "experiment", "model",
    "student_level_auc",
    "institution_heldout_auc_mean", "institution_heldout_auc_std",
    "delta_auc_student_minus_heldout",
]
print("Paper comparison (AUC):")
display(comparison[display_cols])
comparison


## 5. Export CSVs + draft reviewer text

In [ ]:
per_path = OUT / "institution_heldout_per_fold.csv"
sum_path = OUT / "institution_heldout_summary.csv"
cmp_path = OUT / "table_institution_vs_student_level.csv"
per_fold.to_csv(per_path, index=False)
summary.to_csv(sum_path, index=False)
comparison.to_csv(cmp_path, index=False)
print("Wrote:", per_path)
print("Wrote:", sum_path)
print("Wrote:", cmp_path)

e1 = comparison[comparison["experiment"] == "E1_full"].copy()

def fmt_row(r):
    return (
        f"{r['model']}: student-level AUC={r['student_level_auc']:.3f}; "
        f"institution-held-out AUC={r['institution_heldout_auc_mean']:.3f}±{r['institution_heldout_auc_std']:.3f} "
        f"(delta={r['delta_auc_student_minus_heldout']:+.3f})"
    )

lines = [fmt_row(r) for _, r in e1.iterrows()]
mean_delta = float(e1["delta_auc_student_minus_heldout"].mean())

methods = (
    "In addition to the primary student-level stratified 80/10/10 split, we assessed "
    "cross-institution generalization using 5-fold GroupKFold grouped by campus identifier "
    f"(`schoolnum`; N={n_schools} institutions). Within each fold, preprocessing and model "
    "fitting were performed only on training campuses, and evaluation used entirely held-out campuses. "
    "Campus ID was used solely as a grouping key and was not included as a predictor."
)

results = (
    "Institution-held-out discrimination remained close to student-level performance. "
    + " ".join(lines) + " "
    f"Mean AUC drop vs student-level holdout was {mean_delta:+.3f}."
)

rebuttal = (
    "We thank the reviewer for requesting an institution-held-out evaluation. "
    "Our primary evaluation remains the student-level stratified holdout for comparability with the original manuscript. "
    "Because HMS is multi-institutional, we additionally performed leave-institution-out validation via "
    f"GroupKFold on `schoolnum` (K=5; {n_schools} campuses). "
    + results + " "
    "We now report this as a transportability sensitivity analysis (Table: institution vs student-level) "
    "and have updated Methods/Results/Limitations accordingly. "
    "We also report the E6 (no institution covariates) GroupKFold setting as a stronger check that "
    "performance does not rely on institutional characteristic features."
)

print("\n--- Methods (paste) ---\n")
print(methods)
print("\n--- Results (paste) ---\n")
print(results)
print("\n--- Reviewer rebuttal (paste) ---\n")
print(rebuttal)

(OUT / "institution_heldout_manuscript_snippets.md").write_text(
    "# Institution-held-out — manuscript / rebuttal snippets\n\n"
    "## Methods\n\n" + methods + "\n\n"
    "## Results\n\n" + results + "\n\n"
    "## Reviewer rebuttal\n\n" + rebuttal + "\n",
    encoding="utf-8",
)
print("\nWrote:", OUT / "institution_heldout_manuscript_snippets.md")
